# Setup (Google Colab)

Run `data_and_training.ipynb` first so `models/` exists on Drive. Enable **GPU** runtime if available.

In [ ]:
from pathlib import Path

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/DeepL-Final')
else:
    PROJECT_ROOT = Path('.').resolve()

DATA_DIR = PROJECT_ROOT / 'Data' / 'caption_data'
IMAGES_DIR = DATA_DIR / 'Images'
CAPTIONS_FILE = DATA_DIR / 'captions.txt'
MODEL_DIR = PROJECT_ROOT / 'models'

assert (MODEL_DIR / 'caption_model.pt').exists(), 'Train the model in data_and_training.ipynb first.'


ModuleNotFoundError: No module named 'google'

In [ ]:
import json
import random
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


In [ ]:
class AttentionCaptionModel(nn.Module):
    """ResNet-50 encoder + Bahdanau attention + gated LSTM decoder (Show, Attend and Tell)."""

    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        encoder_dim,
        pad_idx,
        dropout=0.5,
        encoder_name='resnet50',
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.hidden_dim = hidden_dim
        self.encoder_dim = encoder_dim

        if encoder_name == 'resnet50':
            backbone = models.resnet50(weights=None)
        elif encoder_name == 'resnet18':
            backbone = models.resnet18(weights=None)
            encoder_dim = 512
            self.encoder_dim = 512
        else:
            raise ValueError(f'Unsupported encoder: {encoder_name}')

        self.encoder = nn.Sequential(*list(backbone.children())[:-2])
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))

        self.init_h = nn.Linear(encoder_dim, hidden_dim)
        self.init_c = nn.Linear(encoder_dim, hidden_dim)

        self.attention_enc = nn.Linear(encoder_dim, hidden_dim, bias=False)
        self.attention_dec = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.attention_full = nn.Linear(hidden_dim, 1)
        self.beta_gate = nn.Linear(hidden_dim, 1)  # gating scalar β = σ(·)

        self.context_proj = nn.Linear(encoder_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm_cell = nn.LSTMCell(embed_dim + hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def encode_image(self, images):
        features = self.encoder(images)
        features = self.adaptive_pool(features)
        batch_size, channels, height, width = features.shape
        features = features.view(batch_size, channels, height * width).permute(0, 2, 1)
        mean_features = features.mean(dim=1)
        h = torch.tanh(self.init_h(mean_features))
        c = torch.tanh(self.init_c(mean_features))
        return features, h, c

    def attention(self, encoder_out, hidden):
        enc = self.attention_enc(encoder_out)
        dec = self.attention_dec(hidden).unsqueeze(1)
        energy = torch.tanh(enc + dec)
        weights = torch.softmax(self.attention_full(energy).squeeze(-1), dim=1)
        context = (encoder_out * weights.unsqueeze(-1)).sum(dim=1)
        beta = torch.sigmoid(self.beta_gate(hidden))
        context = beta * context
        return self.context_proj(context), weights, beta

    def decode_step(self, encoder_out, prev_tokens, hidden, cell, return_attention=False):
        embeddings = self.dropout(self.embedding(prev_tokens))
        context, attn_weights, beta = self.attention(encoder_out, hidden)
        lstm_input = torch.cat([embeddings, context], dim=1)
        hidden, cell = self.lstm_cell(lstm_input, (hidden, cell))
        logits = self.fc(self.dropout(hidden))
        if return_attention:
            return logits, hidden, cell, attn_weights, beta
        return logits, hidden, cell

    def forward(self, images, captions, return_attention=False):
        encoder_out, hidden, cell = self.encode_image(images)
        logits = []
        attn_weights = []
        seq_len = captions.size(1)
        for t in range(seq_len):
            if return_attention:
                step_logits, hidden, cell, weights, _ = self.decode_step(
                    encoder_out, captions[:, t], hidden, cell, return_attention=True
                )
                attn_weights.append(weights.unsqueeze(1))
            else:
                step_logits, hidden, cell = self.decode_step(
                    encoder_out, captions[:, t], hidden, cell
                )
            logits.append(step_logits.unsqueeze(1))
        logits = torch.cat(logits, dim=1)
        if return_attention:
            return logits, torch.cat(attn_weights, dim=1)
        return logits


class ImageCaptionModel(nn.Module):
    """Legacy ResNet18 + LSTM model (backward compatibility)."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_idx):
        super().__init__()
        self.pad_idx = pad_idx
        self.hidden_dim = hidden_dim

        resnet = models.resnet18(weights=None)
        modules = list(resnet.children())[:-1]
        self.encoder = nn.Sequential(*modules)
        self.feature_dim = 512

        self.init_h = nn.Linear(self.feature_dim, hidden_dim)
        self.init_c = nn.Linear(self.feature_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def encode_image(self, images):
        features = self.encoder(images)
        features = features.view(features.size(0), -1)
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        return h0, c0

    def forward(self, images, captions):
        h0, c0 = self.encode_image(images)
        embeddings = self.dropout(self.embedding(captions))
        outputs, _ = self.lstm(embeddings, (h0, c0))
        logits = self.fc(self.dropout(outputs))
        return logits


In [ ]:
def tokenize_caption(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9 ]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()


def word_overlap_score(generated, references):
    generated_words = set(tokenize_caption(generated))
    best = 0.0
    for reference in references:
        reference_words = set(tokenize_caption(reference))
        if not reference_words:
            continue
        best = max(best, len(generated_words & reference_words) / len(reference_words))
    return best


def build_eval_transform(config):
    size = config['image_size']
    mean = config['normalize_mean']
    std = config['normalize_std']

    if config.get('eval_center_crop', False):
        resize = config.get('eval_resize', size + 32)
        return transforms.Compose([
            transforms.Resize(resize),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std),
        ])

    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std),
    ])


def load_model_artifacts(model_dir=MODEL_DIR):
    model_dir = Path(model_dir)
    with open(model_dir / 'config.json', 'r', encoding='utf-8') as f:
        config = json.load(f)
    with open(model_dir / 'vocabulary.json', 'r', encoding='utf-8') as f:
        vocab_data = json.load(f)

    history_path = model_dir / 'training_history.json'
    history = None
    if history_path.exists():
        with open(history_path, 'r', encoding='utf-8') as f:
            history = json.load(f)

    word2idx = vocab_data['word2idx']
    idx2word = {int(k): v for k, v in vocab_data['idx2word'].items()}

    model_type = config.get('model_type', 'lstm')
    if model_type == 'attention_lstm':
        net = AttentionCaptionModel(
            vocab_size=config['vocab_size'],
            embed_dim=config['embed_dim'],
            hidden_dim=config['hidden_dim'],
            encoder_dim=config.get('encoder_dim', 2048),
            pad_idx=config['pad_idx'],
            dropout=config.get('dropout', 0.5),
            encoder_name=config.get('encoder_name', 'resnet50'),
        )
    else:
        net = ImageCaptionModel(
            vocab_size=config['vocab_size'],
            embed_dim=config['embed_dim'],
            hidden_dim=config['hidden_dim'],
            pad_idx=config['pad_idx'],
        )

    ckpt_path = model_dir / 'caption_model.pt'
    try:
        state_dict = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    except TypeError:
        state_dict = torch.load(ckpt_path, map_location=DEVICE)
    net.load_state_dict(state_dict, strict=True)
    net.to(DEVICE)
    net.eval()

    print(f"Loaded model: {model_type} | encoder={config.get('encoder_name', 'resnet18')}")
    if config.get('gated_attention'):
        print('Gated attention: enabled (β gate)')
    if 'beam_width' in config:
        print(f"Beam width: {config['beam_width']}")
    if history and 'test_bleu4' in history:
        print(f"Training BLEU-4 (beam): {history['test_bleu4'] * 100:.2f}")
        if 'test_bleu4_greedy' in history:
            print(f"Training BLEU-4 (greedy): {history['test_bleu4_greedy'] * 100:.2f}")

    return {
        'model': net,
        'word2idx': word2idx,
        'idx2word': idx2word,
        'config': config,
        'transform': build_eval_transform(config),
        'model_type': model_type,
        'history': history,
    }


class TrainedCaptionModel:
    def __init__(self, model_dir=MODEL_DIR):
        self.artifacts = load_model_artifacts(model_dir)


@torch.no_grad()
def generate_greedy(net, image_tensor, word2idx, idx2word, config, max_len=None, return_attention=False):
    net.eval()
    start_idx = word2idx[config['start_token']]
    end_idx = word2idx[config['end_token']]
    max_len = max_len or config['max_seq_len']

    encoder_out, hidden, cell = net.encode_image(image_tensor)
    current = torch.full((1,), start_idx, dtype=torch.long, device=image_tensor.device)
    words = []
    attn_history = []

    for _ in range(max_len):
        if return_attention:
            logits, hidden, cell, attn_w, _ = net.decode_step(
                encoder_out, current, hidden, cell, return_attention=True
            )
            attn_history.append(attn_w.squeeze(0).cpu())
        else:
            logits, hidden, cell = net.decode_step(encoder_out, current, hidden, cell)
        next_idx = logits.argmax(dim=-1)
        token_id = next_idx.item()
        if token_id == end_idx:
            break
        words.append(idx2word[token_id])
        current = next_idx

    caption = ' '.join(words)
    if return_attention:
        return caption, attn_history
    return caption


@torch.no_grad()
def generate_beam(net, image_tensor, word2idx, idx2word, config, beam_width=None, max_len=None):
    """Beam search decoding — typically improves BLEU-4 over greedy."""
    net.eval()
    start_idx = word2idx[config['start_token']]
    end_idx = word2idx[config['end_token']]
    beam_width = beam_width or config.get('beam_width', 5)
    max_len = max_len or config['max_seq_len']

    encoder_out, hidden, cell = net.encode_image(image_tensor)
    beams = [(0.0, [start_idx], hidden, cell)]
    completed = []

    for _ in range(max_len):
        candidates = []
        for log_prob, tokens, h, c in beams:
            if tokens[-1] == end_idx:
                completed.append((log_prob, tokens))
                continue

            current = torch.tensor([tokens[-1]], device=image_tensor.device)
            logits, new_h, new_c = net.decode_step(encoder_out, current, h, c)
            log_probs = torch.log_softmax(logits, dim=-1).squeeze(0)

            topk_log_probs, topk_ids = log_probs.topk(beam_width)
            for lp, tid in zip(topk_log_probs.tolist(), topk_ids.tolist()):
                candidates.append((log_prob + lp, tokens + [tid], new_h, new_c))

        if not candidates:
            break

        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]

        if all(b[1][-1] == end_idx for b in beams):
            completed.extend((b[0], b[1]) for b in beams)
            break

    if completed:
        best = max(completed, key=lambda x: x[0] / max(len(x[1]), 1))
    else:
        best = max(beams, key=lambda x: x[0] / max(len(x[1]), 1))
        best = (best[0], best[1])

    words = [idx2word[t] for t in best[1][1:] if t not in (start_idx, end_idx)]
    return ' '.join(words)


def visualize_attention(image_path, caption, attn_history, grid_size=7):
    """Overlay 7×7 attention maps on the image for each generated word."""
    pil_image = Image.open(image_path).convert('RGB')
    words = caption.split()
    n_steps = min(len(words), len(attn_history))
    if n_steps == 0:
        print('No attention weights to visualize.')
        return

    cols = min(4, n_steps)
    rows = (n_steps + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.atleast_1d(axes).flatten()

    for i in range(n_steps):
        attn = attn_history[i].numpy().reshape(grid_size, grid_size)
        attn = attn / max(attn.max(), 1e-8)
        attn_up = np.array(
            Image.fromarray((attn * 255).astype(np.uint8)).resize(
                pil_image.size, Image.Resampling.BILINEAR
            )
        ) / 255.0

        ax = axes[i]
        ax.imshow(pil_image)
        ax.imshow(attn_up, cmap='jet', alpha=0.45)
        ax.set_title(words[i], fontsize=10)
        ax.axis('off')

    for j in range(n_steps, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Attention maps per generated word', fontsize=12)
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def generate_caption(image_path, model, max_len=None, use_beam=True, return_attention=False):
    if isinstance(model, (str, Path)):
        artifacts = load_model_artifacts(Path(model))
    elif isinstance(model, dict):
        artifacts = model
    elif hasattr(model, 'artifacts'):
        artifacts = model.artifacts
    else:
        raise ValueError('model must be TrainedCaptionModel, artifacts dict, or model directory path')

    net = artifacts['model']
    word2idx = artifacts['word2idx']
    idx2word = artifacts['idx2word']
    config = artifacts['config']
    transform = artifacts['transform']
    model_type = artifacts.get('model_type', config.get('model_type', 'lstm'))

    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(DEVICE)
    net.eval()

    if model_type == 'attention_lstm':
        if return_attention:
            if use_beam:
                print('Note: attention maps use greedy decoding (beam search does not retain per-step weights).')
            return generate_greedy(
                net, image_tensor, word2idx, idx2word, config,
                max_len=max_len, return_attention=True,
            )
        if use_beam:
            return generate_beam(net, image_tensor, word2idx, idx2word, config, max_len=max_len)
        return generate_greedy(net, image_tensor, word2idx, idx2word, config, max_len=max_len)

    # Legacy LSTM model
    start_idx = word2idx[config['start_token']]
    end_idx = word2idx[config['end_token']]
    max_len = max_len or config['max_seq_len']
    words = []
    h, c = net.encode_image(image_tensor)
    current = torch.tensor([[start_idx]], device=DEVICE)
    for _ in range(max_len):
        embedding = net.embedding(current)
        output, (h, c) = net.lstm(embedding, (h, c))
        logits = net.fc(output.squeeze(1))
        next_idx = logits.argmax(dim=-1).item()
        if next_idx == end_idx:
            break
        words.append(idx2word[next_idx])
        current = torch.tensor([[next_idx]], device=DEVICE)
    return ' '.join(words)


# Demonstration

In [ ]:
model = TrainedCaptionModel()
artifacts = model.artifacts
config = artifacts['config']
transform = artifacts['transform']

with open(MODEL_DIR / 'data_splits.json', 'r', encoding='utf-8') as f:
    splits = json.load(f)

test_images = splits['test_images']
caption_df = pd.read_csv(CAPTIONS_FILE)
reference_map = caption_df.groupby('image')['caption'].apply(list).to_dict()

random.seed(splits['seed'])
demo_images = random.sample(test_images, k=min(8, len(test_images)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for ax, image_name in zip(axes, demo_images):
    image_path = IMAGES_DIR / image_name
    image_tensor = transform(Image.open(image_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    beam_cap = generate_beam(
        artifacts['model'], image_tensor,
        artifacts['word2idx'], artifacts['idx2word'], config,
    )
    greedy_cap = generate_greedy(
        artifacts['model'], image_tensor,
        artifacts['word2idx'], artifacts['idx2word'], config,
    )
    references = reference_map[image_name][:2]

    ax.imshow(Image.open(image_path).convert('RGB'))
    ax.axis('off')
    ax.set_title(
        f'Beam: {beam_cap}\nGreedy: {greedy_cap}\nRef: {" | ".join(references)}',
        fontsize=7,
    )

plt.suptitle('Test set captions — beam vs greedy', fontsize=14)
plt.tight_layout()
plt.show()


# Analysis

In [ ]:
attn_demo_image = demo_images[1]
attn_image_path = IMAGES_DIR / attn_demo_image
attn_tensor = transform(Image.open(attn_image_path).convert('RGB')).unsqueeze(0).to(DEVICE)

attn_caption, attn_history = generate_greedy(
    artifacts['model'], attn_tensor,
    artifacts['word2idx'], artifacts['idx2word'], config,
    return_attention=True,
)

print(f'Image: {attn_demo_image}')
print(f'Generated (greedy): {attn_caption}')
print(f'References: {reference_map[attn_demo_image][:2]}')
visualize_attention(attn_image_path, attn_caption, attn_history)


In [ ]:
eval_images = random.sample(test_images, k=min(120, len(test_images)))
results = []

for image_name in eval_images:
    image_path = IMAGES_DIR / image_name
    generated = generate_caption(str(image_path), model, use_beam=True)
    references = reference_map[image_name]
    score = word_overlap_score(generated, references)
    results.append({
        'image': image_name,
        'generated': generated,
        'references': references,
        'score': score,
    })

results = sorted(results, key=lambda item: item['score'], reverse=True)
success_cases = results[:3]
failure_cases = results[-3:]

print(f'Mean word-overlap on {len(eval_images)} test images (beam): {np.mean([r["score"] for r in results]):.3f}')
print()
print('Successful examples')
for case in success_cases:
    print(f"Image: {case['image']}")
    print(f"Generated: {case['generated']}")
    print(f"Reference: {case['references'][0]}")
    print(f"Word overlap score: {case['score']:.2f}")
    print()

print('Failure examples')
for case in failure_cases:
    print(f"Image: {case['image']}")
    print(f"Generated: {case['generated']}")
    print(f"Reference: {case['references'][0]}")
    print(f"Word overlap score: {case['score']:.2f}")
    print()

In [ ]:
def show_case_grid(cases, title):
    fig, axes = plt.subplots(1, len(cases), figsize=(5 * len(cases), 5))
    if len(cases) == 1:
        axes = [axes]
    for ax, case in zip(axes, cases):
        image_path = IMAGES_DIR / case['image']
        ax.imshow(Image.open(image_path).convert('RGB'))
        ax.axis('off')
        ax.set_title(
            f"Generated: {case['generated']}\nReference: {case['references'][0]}",
            fontsize=9,
        )
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


show_case_grid(success_cases, 'Successful captioning cases')
show_case_grid(failure_cases, 'Failure captioning cases')